# 07 Calibration Evaluation

This notebook is the main numerical experiment results notebook.

It evaluates probabilistic VAR forecasts generated in `06_forecasts.ipynb` across:

- four controlled DGPs,
- four innovation models,
- multiple calibration and scoring metrics.

The central question is:

**When do flexible innovation models materially improve probabilistic forecast calibration under innovation misspecification?**

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.seeds import set_seed
from src.utils.logging_utils import make_run_dir, save_metrics_json, save_table_csv
from src.data.loaders import load_npz

from src.evaluation.metrics import (
    summarize_probabilistic_forecast,
    make_summary_row,
)

from src.calibration.reliability import reliability_table
from src.calibration.pit import (
    pit_values_multivariate_marginal,
    pit_histogram,
    pit_deviation_from_uniform,
)

from src.calibration.ece import calibration_curve_data

set_seed(123)

In [2]:
dgp_names = [
    "gaussian",
    "student_t",
    "mixture",
    "heteroskedastic",
]

innovation_model_names = [
    "gaussian",
    "bootstrap",
    "student_t",
    "diffusion",
]

forecast_model = "VAR"

forecast_dir = ROOT / "results" / "forecasts" / "06_forecasts"
table_dir = ROOT / "results" / "tables" / "07_calibration_evaluation"
figure_dir = ROOT / "results" / "figures" / "07_calibration_evaluation"

table_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

interval = (0.05, 0.95)
nominal_levels = (0.5, 0.8, 0.9)

In [3]:
forecast_store = {}

for dgp_name in dgp_names:
    forecast_store[dgp_name] = {}

    for innovation_model in innovation_model_names:
        path = forecast_dir / f"{dgp_name}_{innovation_model}_forecast_paths.npz"

        data = load_npz(path)

        forecast_store[dgp_name][innovation_model] = {
            "forecast_paths": data["forecast_paths"],
            "innovation_paths": data["innovation_paths"],
            "y_true": data["y_true"],
        }

        print(
            dgp_name,
            innovation_model,
            data["forecast_paths"].shape,
            data["y_true"].shape,
        )

gaussian gaussian (250, 40, 3) (40, 3)
gaussian bootstrap (250, 40, 3) (40, 3)
gaussian student_t (250, 40, 3) (40, 3)
gaussian diffusion (250, 40, 3) (40, 3)
student_t gaussian (250, 40, 3) (40, 3)
student_t bootstrap (250, 40, 3) (40, 3)
student_t student_t (250, 40, 3) (40, 3)
student_t diffusion (250, 40, 3) (40, 3)
mixture gaussian (250, 40, 3) (40, 3)
mixture bootstrap (250, 40, 3) (40, 3)
mixture student_t (250, 40, 3) (40, 3)
mixture diffusion (250, 40, 3) (40, 3)
heteroskedastic gaussian (250, 40, 3) (40, 3)
heteroskedastic bootstrap (250, 40, 3) (40, 3)
heteroskedastic student_t (250, 40, 3) (40, 3)
heteroskedastic diffusion (250, 40, 3) (40, 3)


## Main Forecast Evaluation Table

The first evaluation step computes a unified set of probabilistic forecast metrics for each DGP and innovation model.

Metrics include:

- empirical interval coverage,
- average interval width,
- expected calibration error,
- PIT deviation from uniformity,
- CRPS,
- energy score,
- interval score.

Lower is better for ECE, PIT deviation, CRPS, energy score, and interval score. Coverage should be close to the nominal level.

In [4]:
summary_rows = []
summary_objects = {}

for dgp_name in dgp_names:
    summary_objects[dgp_name] = {}

    for innovation_model in innovation_model_names:
        obj = forecast_store[dgp_name][innovation_model]

        paths = obj["forecast_paths"]
        y_true = obj["y_true"]

        summary = summarize_probabilistic_forecast(
            forecast_paths=paths,
            y_true=y_true,
            interval=interval,
            nominal_levels=nominal_levels,
        )

        summary_objects[dgp_name][innovation_model] = summary

        row = make_summary_row(
            dgp_name=dgp_name,
            forecast_model=forecast_model,
            innovation_model=innovation_model,
            summary=summary,
        )

        summary_rows.append(row)

main_results_df = pd.DataFrame(summary_rows)

main_results_df

,dgp,forecast_model,innovation_model,avg_coverage,avg_width,energy_score,crps,interval_score,ece,pit_deviation,coverage_1,width_1,coverage_2,width_2,coverage_3,width_3
0,gaussian,VAR,gaussian,0.908333,3.730880,1.301017,0.651583,5.165528,0.019444,0.028333,0.850,4.047602,0.975,3.524876,0.900,3.620163
1,gaussian,VAR,bootstrap,0.900000,3.663531,1.287669,0.641501,5.065912,0.008333,0.033333,0.875,3.943443,0.975,3.552684,0.850,3.494465
2,gaussian,VAR,student_t,0.908333,3.526298,1.292401,0.643233,5.083660,0.022222,0.033333,0.875,3.861631,0.975,3.298324,0.875,3.418940
3,gaussian,VAR,diffusion,0.883333,3.661504,1.277003,0.636873,5.043519,0.013889,0.033333,0.875,4.038879,0.925,3.342484,0.850,3.603148
4,student_t,VAR,gaussian,0.791667,5.108414,2.837641,1.434337,15.662288,0.080556,0.025000,0.725,5.621880,0.800,5.052529,0.850,4.650834
5,student_t,VAR,bootstrap,0.741667,4.760542,2.871302,1.450165,16.258517,0.152778,0.036667,0.700,5.291312,0.775,4.749165,0.750,4.241148
6,student_t,VAR,student_t,0.791667,4.828365,2.861246,1.448419,15.890231,0.108333,0.025000,0.775,5.365158,0.800,4.748382,0.800,4.371555
7,student_t,VAR,diffusion,0.775000,5.120182,2.827546,1.437373,15.473150,0.116667,0.028333,0.750,5.755243,0.825,4.995884,0.750,4.609418
8,mixture,VAR,gaussian,0.875000,5.266876,2.028862,1.000025,9.174847,0.036111,0.030000,0.875,5.632926,0.825,5.151333,0.925,5.016368
9,mixture,VAR,bootstrap,0.866667,4.941106,2.050558,1.007908,9.128005,0.044444,0.026667,0.875,5.532775,0.800,4.911664,0.925,4.378880


In [5]:
nominal_coverage = interval[1] - interval[0]

main_results_df["nominal_coverage"] = nominal_coverage
main_results_df["coverage_error"] = (
    main_results_df["avg_coverage"] - nominal_coverage
)
main_results_df["abs_coverage_error"] = (
    main_results_df["coverage_error"].abs()
)

main_results_df = main_results_df.sort_values(
    ["dgp", "ece"]
)

main_results_df

,dgp,forecast_model,innovation_model,avg_coverage,avg_width,energy_score,crps,interval_score,ece,pit_deviation,coverage_1,width_1,coverage_2,width_2,coverage_3,width_3,nominal_coverage,coverage_error,abs_coverage_error
1,gaussian,VAR,bootstrap,0.900000,3.663531,1.287669,0.641501,5.065912,0.008333,0.033333,0.875,3.943443,0.975,3.552684,0.850,3.494465,0.9,1.110223e-16,1.110223e-16
3,gaussian,VAR,diffusion,0.883333,3.661504,1.277003,0.636873,5.043519,0.013889,0.033333,0.875,4.038879,0.925,3.342484,0.850,3.603148,0.9,-1.666667e-02,1.666667e-02
0,gaussian,VAR,gaussian,0.908333,3.730880,1.301017,0.651583,5.165528,0.019444,0.028333,0.850,4.047602,0.975,3.524876,0.900,3.620163,0.9,8.333333e-03,8.333333e-03
2,gaussian,VAR,student_t,0.908333,3.526298,1.292401,0.643233,5.083660,0.022222,0.033333,0.875,3.861631,0.975,3.298324,0.875,3.418940,0.9,8.333333e-03,8.333333e-03
13,heteroskedastic,VAR,bootstrap,1.000000,6.932733,0.871850,0.426560,6.932733,0.236111,0.090000,1.000,7.607522,1.000,6.417622,1.000,6.773056,0.9,1.000000e-01,1.000000e-01
15,heteroskedastic,VAR,diffusion,1.000000,6.811375,0.861288,0.424650,6.811375,0.250000,0.093333,1.000,7.426766,1.000,6.666014,1.000,6.341344,0.9,1.000000e-01,1.000000e-01
14,heteroskedastic,VAR,student_t,1.000000,6.264756,0.956414,0.475668,6.264756,0.258333,0.100000,1.000,7.049229,1.000,5.840789,1.000,5.904251,0.9,1.000000e-01,1.000000e-01
12,heteroskedastic,VAR,gaussian,1.000000,6.661520,1.060416,0.529416,6.661520,0.263889,0.111667,1.000,7.426244,1.000,6.217772,1.000,6.340544,0.9,1.000000e-01,1.000000e-01
11,mixture,VAR,diffusion,0.883333,5.522463,2.024549,0.993698,8.955996,0.019444,0.030000,0.875,6.098314,0.850,5.409904,0.925,5.059170,0.9,-1.666667e-02,1.666667e-02
10,mixture,VAR,student_t,0.866667,4.948211,2.014931,0.992789,9.048722,0.025000,0.018333,0.875,5.289432,0.825,4.816956,0.900,4.738246,0.9,-3.333333e-02,3.333333e-02


In [ ]:
save_table_csv(
    main_results_df,
    table_dir / "main_calibration_results.csv",
)

main_results_df

In [6]:
display_cols = [
    "dgp",
    "innovation_model",
    "avg_coverage",
    "abs_coverage_error",
    "avg_width",
    "ece",
    "pit_deviation",
    "crps",
    "energy_score",
    "interval_score",
]

compact_results_df = main_results_df[display_cols].copy()

compact_results_df

,dgp,innovation_model,avg_coverage,abs_coverage_error,avg_width,ece,pit_deviation,crps,energy_score,interval_score
1,gaussian,bootstrap,0.900000,1.110223e-16,3.663531,0.008333,0.033333,0.641501,1.287669,5.065912
3,gaussian,diffusion,0.883333,1.666667e-02,3.661504,0.013889,0.033333,0.636873,1.277003,5.043519
0,gaussian,gaussian,0.908333,8.333333e-03,3.730880,0.019444,0.028333,0.651583,1.301017,5.165528
2,gaussian,student_t,0.908333,8.333333e-03,3.526298,0.022222,0.033333,0.643233,1.292401,5.083660
13,heteroskedastic,bootstrap,1.000000,1.000000e-01,6.932733,0.236111,0.090000,0.426560,0.871850,6.932733
15,heteroskedastic,diffusion,1.000000,1.000000e-01,6.811375,0.250000,0.093333,0.424650,0.861288,6.811375
14,heteroskedastic,student_t,1.000000,1.000000e-01,6.264756,0.258333,0.100000,0.475668,0.956414,6.264756
12,heteroskedastic,gaussian,1.000000,1.000000e-01,6.661520,0.263889,0.111667,0.529416,1.060416,6.661520
11,mixture,diffusion,0.883333,1.666667e-02,5.522463,0.019444,0.030000,0.993698,2.024549,8.955996
10,mixture,student_t,0.866667,3.333333e-02,4.948211,0.025000,0.018333,0.992789,2.014931,9.048722


In [ ]:
save_table_csv(
    compact_results_df,
    table_dir / "compact_calibration_results.csv",
)